# Reconstructing Panel Figure 1
This notebook can be used to reproduce Panel Figure 1 from the main text of the gVAMP paper.
## Imports

In [ ]:
import pandas as pd
import numpy as np
import string
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

## Size and font

In [ ]:
eps = 1e-32

img_width = 16
img_height = 16

fs_normal = 18
fs_large = 20
fs_small = 16
fw_sublbl = 500
fw_axlbl = 500

## Source data
Specify the path to the source data for the figure here

In [ ]:
Fig1a_source_fpath = "Fig1a_source.csv"
Fig1b_source_fpath = "Fig1b_source.csv"

## Figure 1a

In [ ]:
df = pd.read_table(Fig1a_source_fpath, sep="\t")

rows = df["Row"].unique().astype(int)
labels = df["Label"].unique()
n_rows = len(rows)

# Axis limits
tpr_lims = [0.6, 0.6, 0.5, 0.4]
fdr_lims = [0.5, 0.4, 0.3, 0.1]

colors = ['tab:blue', 'tab:orange', 'tab:green']
offset_y = 0.01
bar_width = 0.1

fig, axs = plt.subplots(n_rows, 1, figsize=(int(img_width * 1 / 4), img_height))
axs = axs.flatten()

for i in rows:
    ax = axs[i]
    for lbl_idx, lbl in enumerate(labels):
        tpr_mean = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "mean")]["TPR"].values
        fdr_mean = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "mean")]["FDR"].values
        ax.plot(fdr_mean, tpr_mean, alpha=0.5, color=colors[lbl_idx % len(colors)], label=lbl)

        tpr_std_upper = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "std_upper")]["TPR"].values
        tpr_std_lower = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "std_lower")]["TPR"].values
        ax.fill_between(fdr_mean, tpr_std_lower, tpr_std_upper, color="grey", alpha=0.2)

        mean_tpr_95 = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "point")]["TPR"].values
        mean_fdr_95 = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "point")]["FDR"].values
        ax.scatter(mean_fdr_95, mean_tpr_95, color=colors[lbl_idx % len(colors)])       

    ax.set_xlim([0, fdr_lims[i]])
    ax.set_ylim([0, tpr_lims[i]])
    ax.vlines(0.05, ymin=0, ymax=1, color="black", linestyles="dashed", label="FDR05", alpha=0.5)
    ax.spines[["right", "top"]].set_visible(False)
    ax.tick_params(axis='x', labelsize=fs_normal)
    ax.tick_params(axis='y', labelsize=fs_normal)

axs[-1].legend(loc="lower right", fontsize=fs_small, frameon=False, handlelength=0.8, bbox_to_anchor=(1.2, 0.00))
fig.supxlabel('FDR', fontsize=fs_large, fontweight=fw_axlbl)
fig.supylabel('TPR', fontsize=fs_large, fontweight=fw_axlbl)
fig.tight_layout()
fig.savefig("Fig1a.png", dpi=300)
plt.show()

## Figure 1b

In [ ]:
df = pd.read_table(Fig1b_source_fpath, sep="\t")

bar_width = 0.1
colors = ['tab:green', 'tab:blue', 'tab:orange', 'tab:red', 'tab:purple']
offset_y = 0.01

bin_labels = df["MAF_bin"].unique()
x = np.arange(len(bin_labels))
rows = df["Row"].unique().astype(int)
labels = df["Label"].unique()

n_rows = len(rows)

fig, axs = plt.subplots(n_rows, 1, figsize=(int(3/4 * img_width), img_height), sharex=True)

for i in rows:
    ax = axs[i]
    for lbl_idx, lbl in enumerate(labels):
        means = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "mean")]["Proportion_of_variance"].values
        stds = df[(df["Row"] == i) & (df["Label"] == lbl) & (df["Property"] == "std")]["Proportion_of_variance"].values
        
        if lbl == "Ground_truth":

            offset = (2 - len(labels) / 2) * bar_width
            
            ax.bar(
                x + offset,
                means,
                yerr=stds,
                width=bar_width,
                color='none',
                edgecolor='red',
                linewidth=2,
                label='Ground truth',
                capsize=3,
                zorder=3,
                error_kw=dict(ecolor='red', lw=1.5)
            )
            
        else:
            offset = (lbl_idx - len(labels) / 2) * bar_width

            ax.bar(x + offset,
                   means, 
                   yerr=stds, 
                   width=bar_width,
                   color=colors[lbl_idx % len(colors)], 
                   alpha=0.5,
                   label=lbl,
                   capsize=3
                  )

    ax.tick_params(axis='x', labelsize=fs_normal)
    ax.tick_params(axis='y', labelsize=fs_normal)
    ax.set_ylim([0,1])
    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels)
    ax.spines[['right', 'top']].set_visible(False)
    if i < n_rows:
        ax.spines['bottom'].set_visible(False)
    ax.grid(alpha=0.1)

axs[0].legend(fontsize=fs_normal, loc='upper right', frameon=False)
fig.supxlabel('MAF bins', fontsize=fs_large, fontweight=fw_axlbl)
fig.supylabel('Proportion of variance', fontsize=fs_large, fontweight=fw_axlbl)
plt.tight_layout()
fig.savefig("Fig1b.png", dpi=300)
plt.show()

## Final Figure 1

In [ ]:
img_fpath_list = ["Fig1a.png", 
                  'Fig1b.png'
                 ]

labels = list(string.ascii_lowercase)[:len(img_fpath_list)]

fig = plt.figure(figsize=(img_width, img_height), dpi=300)
fig.subplots_adjust(hspace=0, wspace=0)
gs = GridSpec(4, 4, wspace=0, hspace=0)
axs = [
    fig.add_subplot(gs[:, 0]),
    fig.add_subplot(gs[:, 1:])
]

for i,ax in enumerate(axs):
    ax.imshow(plt.imread(img_fpath_list[i])) 
    ax.set_axis_off()
    ax.set_title(labels[i], loc='left', fontsize=fs_normal, fontweight=fw_sublbl)

fig.savefig("Fig1.png",dpi=300, bbox_inches='tight')
plt.show()